# Pytorch的自动微分模块

## 1. 自动微分的两个核心信息

每个张量（Tensor）身上挂着两样东西：**值**存在 `.data` 里，**它怎么被算出来的**存在 `.grad_fn` 里。

比如 `c = a + b`，c 的 `.data` 就是加法结果，c 的 `.grad_fn` 记录着"我是通过加法产生的"。反向传播时，靠 `grad_fn` 才知道梯度该沿着什么运算往回传。

## 2. PyTorch 是动态图

- **静态图**（比如 TensorFlow 1.x）：先画好整张计算图，再喂数据跑。就像盖房子先画完图纸再施工。
- **动态图**（PyTorch）：**边算边搭图**。每做一次运算，就在计算图里加一个节点。好处是灵活，代码怎么写图就怎么长，随时可以改。

## 3. requires_grad 是"追踪开关"

- 把某个张量的 `requires_grad=True`，就等于对它说：**"你的梯度很重要，反向传播时要算"**
- 任何从这个张量派生出来的张量，会自动标记为 `requires_grad=True`（因为要算父节点的梯度，子节点也得参与）
- 当你在最终结果（根节点）上调用 `.backward()`，整张图从后往前全自动算一遍梯度

## 4. 中间结果的梯度会被扔掉

反向传播算完梯度后：

- **叶子节点**（你亲手创建的参数，比如 `W`、`b`）：梯度保留，并且**会累加**（每次 `.backward()` 把新梯度加到旧梯度上）
- **中间节点**（运算过程中产生的临时结果）：梯度直接释放，省内存

所以每次训练循环开头都要 `optimizer.zero_grad()`——把上次的梯度清零，不然会越加越多。

如果想保留中间节点的梯度，可以设置 `retain_grad=True`。

In [1]:
import torch

In [3]:
# 定义数据
x = torch.tensor(10.0)
y = torch.tensor(3.0)
print(x)
print(y)

tensor(10.)
tensor(3.)


In [5]:
# 初始化参数
w = torch.rand(1,1,requires_grad=True)
b = torch.rand(1,1,requires_grad=True)

In [6]:
# 前向传播计算输出值Z
z = w * x + b
print(z)

tensor([[9.5769]], grad_fn=<AddBackward0>)


### 叶子节点和根结点
- 在计算图中x，w，b为叶子节点，叶子节点的数据并非由计算生成，因此是整个计算中的基石，叶子节点**不可以**做`in-place`操作
  - 叶子节点在方向传播的时候会记住梯度
- 最终的loss为根结点

In [7]:
print(x.is_leaf)
print(w.is_leaf)
print(b.is_leaf)
print(y.is_leaf)
print(z.is_leaf)

True
True
True
True
False


In [8]:
# 设置损失函数
loss = torch.nn.MSELoss()
loss_value = loss(z,y)
print(loss_value)
print(loss_value.is_leaf)

tensor(43.2553, grad_fn=<MseLossBackward0>)
False


/Users/bowen/miniconda3/envs/ailab/lib/python3.10/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [9]:
# 方向传播
loss_value.backward()


In [10]:
# 查看所有的梯度
print(w.grad)
print(b.grad)

tensor([[131.5376]])
tensor([[13.1538]])
